# Kindle Review

#### Best Practices
- Preprocessing and cleaning
- Train test split
- BOW, TFIDF, Word2Vec
-  Train ML Model

In [29]:
import pandas as pd
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

In [30]:
# load the dataset
df = pd.read_csv('all_kindle_review.csv')

In [31]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [32]:
# we dont require all the columns. We only need reviewText and rating column
df = df[['reviewText', 'rating']]

In [33]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [34]:
df.shape

(12000, 2)

In [35]:
# Checking missing values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [36]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [37]:
df['rating'].value_counts() # so its a balanced dataset

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

#### Preprocessing and Cleaning

In [50]:
# for ratings below three we put it into negative review and above three we put it into positive reviews in the rating column.

## positive review is = and negative review is 1
df['rating'] = df['rating'].apply(lambda x:0 if x<3 else 1)

In [51]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,0
1,great short read didnt want put read one sitti...,0
2,ill start saying first four books wasnt expect...,0
3,aggie angela lansbury carries pocketbooks inst...,0
4,expect type book library pleased find price right,0


In [52]:
df['rating'].unique() # so we converted 1,2 raiting to 1 and 3,4,5 rating as 0

array([0])

In [53]:
df['rating'].value_counts()

rating
0    12000
Name: count, dtype: int64

In [54]:
## Lower all the words in reviewText column

df['reviewText'] = df['reviewText'].str.lower()

In [55]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,0
1,great short read didnt want put read one sitti...,0
2,ill start saying first four books wasnt expect...,0
3,aggie angela lansbury carries pocketbooks inst...,0
4,expect type book library pleased find price right,0


In [56]:
# removing special characters
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub('[^a-z A-Z 0-9]+','',x))

# remove stopwords
stop_words = set(stopwords.words('english'))
df['reviewText'] = df['reviewText'].apply(lambda x:' '.join([y for y in x.split() if y not in stop_words]))

In [57]:
# remove url
df['reviewText'] = df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))(/[\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '', str(x)))


In [58]:
# remove html tags
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

In [59]:
# remove any additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x:' '.join(x.split()))

In [60]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,0
1,great short read didnt want put read one sitti...,0
2,ill start saying first four books wasnt expect...,0
3,aggie angela lansbury carries pocketbooks inst...,0
4,expect type book library pleased find price right,0


#### Lemmatization

In [61]:
from nltk.stem import WordNetLemmatizer

In [62]:
lemmatizer = WordNetLemmatizer()

In [63]:
def lemmatize_words(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

In [64]:
df['reviewText'] = df['reviewText'].apply(lambda x: lemmatize_words(x))

In [65]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,0
1,great short read didnt want put read one sitti...,0
2,ill start saying first four book wasnt expecti...,0
3,aggie angela lansbury carry pocketbook instead...,0
4,expect type book library pleased find price right,0


#### Train test split

In [66]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.20)